# 04 — Run combined VAE + univariate AoU replication models

Proteomics and clinical chemistries are fit separately, but through one shared, PC-adjusted model engine.

## Two complementary AoU estimands

1. **Common main-effect model for every discovery method**

   `z(omic feature) ~ disease case + spline(age) + sex + PC1–PC8`

   This gives a directly comparable case coefficient for VAE, OLS, and spline-discovery targets.

2. **Discovery-concordant model for spline-discovery targets**

   `z(omic feature) ~ disease case * spline(age) + sex + PC1–PC8`

   The prespecified replication test is a joint HC3 Wald test of all disease × spline(age) terms. Fixed age-specific contrasts at ages 40, 55, and 70 are generated only for interpretation and plotting; their individual p-values are **not** used to choose the most favorable age.

Both models use domain-specific controls in the primary analysis. Sensitivity models use clean controls, linear age, age strata, and a no-PC comparison. The workflow never chooses the smallest p-value across sensitivities.

FDR is calculated on unique AoU tests, not duplicated discovery rows. Every output retains explicit test-family, model-status, denominator, and feature-link identifiers for troubleshooting.


In [ ]:
from pathlib import Path
import os
import sys

# Jupyter's file-browser directory and the kernel working directory can differ.
# Locate the project by its marker instead of trusting Path.cwd().
def find_project_root():
    marker_name = ".aou_combined_replication"
    project_name = "Combined_VAE_linear_AoU_replication"

    env_root = os.getenv("AOU_COMBINED_REPLICATION_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / marker_name).exists():
            return root
        raise FileNotFoundError(
            f"AOU_COMBINED_REPLICATION_ROOT={root} lacks {marker_name}"
        )

    here = Path.cwd().resolve()
    ancestors = [here, *here.parents]
    for candidate in ancestors:
        if (candidate / marker_name).exists():
            return candidate

    matches = []
    for parent in ancestors[:5]:
        patterns = [
            f"*/{marker_name}",
            f"{project_name}/{marker_name}",
            f"*/{project_name}/{marker_name}",
        ]
        for pattern in patterns:
            matches.extend(path.parent for path in parent.glob(pattern))

    matches = sorted(set(path.resolve() for path in matches))
    named = [path for path in matches if path.name == project_name]
    matches = named or matches
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(
        "Could not locate the combined replication project. Set "
        "AOU_COMBINED_REPLICATION_ROOT explicitly. Candidates found: "
        + str(matches)
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import project_config as cfg
from src import aou_replication_utils as aru

print("Kernel working directory:", Path.cwd().resolve())
print("Project root:", cfg.PROJECT_ROOT)
print("Workspace root:", cfg.WORKSPACE_ROOT)


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

targets_file = cfg.TARGET_DIR / "analysis_targets_grouped.tsv"
model_plan_file = cfg.CONFIG_DIR / "model_plan.tsv"
protein_matrix_file = cfg.PROTEOMICS_DIR / "proteomics_analysis_matrix.tsv.gz"
protein_map_file = cfg.PROTEOMICS_DIR / "proteomics_feature_column_map.tsv"
chemistry_matrix_file = cfg.CHEMISTRY_DIR / "chemistry_analysis_matrix.tsv.gz"
chemistry_map_file = cfg.CHEMISTRY_DIR / "chemistry_feature_column_map.tsv"

required = [
    targets_file,
    model_plan_file,
    protein_matrix_file,
    protein_map_file,
    chemistry_matrix_file,
    chemistry_map_file,
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required files. Run notebooks 00-03 first. Missing: "
        + "; ".join(str(path) for path in missing)
    )

targets = pd.read_csv(targets_file, sep="\t")
model_plan = aru.load_model_plan(model_plan_file)

# The discovery-concordant spline interaction must use one unambiguous primary
# specification. It inherits the primary control set, sex adjustment, and PCs.
primary_rows = model_plan[model_plan["is_primary"]].copy()
if len(primary_rows) != 1:
    raise ValueError(
        "model_plan.tsv must contain exactly one is_primary=TRUE row; "
        f"found {len(primary_rows)}"
    )
primary_spec = primary_rows.iloc[0].to_dict()
if primary_spec["age_form"] != "spline" or primary_spec["age_subset"] != "all":
    raise ValueError(
        "The primary model must use age_form='spline' and age_subset='all' "
        "for the discovery-concordant spline interaction test."
    )

print(f"Grouped discovery targets: {len(targets):,}")
print("Primary model:", primary_spec["analysis_label"])
display(model_plan)


In [ ]:
def load_feature_map(path):
    """Load reviewed feature-to-matrix-column links.

    Repeated feature keys are expected when one protein occurs in multiple
    discovery rows. They are collapsed when every occurrence maps to the
    same outcome column.

    The function stops only when one feature key maps to more than one
    distinct outcome column, because choosing among conflicting assay
    columns automatically would be scientifically unsafe.
    """

    path = Path(path)

    table = pd.read_csv(
        path,
        sep="\t",
        dtype="string",
    )

    required_columns = {
        "feature_key",
        "available_for_model",
        "outcome_col",
    }

    missing_columns = sorted(
        required_columns - set(table.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Feature map lacks required columns: {path}\n"
            f"Missing: {missing_columns}\n"
            f"Available: {table.columns.tolist()}"
        )

    # Normalize values before evaluating availability and uniqueness.
    table["feature_key"] = (
        table["feature_key"]
        .str.strip()
        .replace("", pd.NA)
    )

    table["outcome_col"] = (
        table["outcome_col"]
        .str.strip()
        .replace("", pd.NA)
    )

    availability_text = (
        table["available_for_model"]
        .str.strip()
        .str.lower()
    )

    available = availability_text.isin(
        ["true", "1", "yes", "y", "t"]
    )

    usable = table.loc[
        available
        & table["feature_key"].notna()
        & table["outcome_col"].notna()
    ].copy()

    # The same feature-target mapping may occur multiple times because the
    # protein was selected by more than one discovery method or PRS.
    before_exact_dedup = len(usable)

    usable = usable.drop_duplicates(
        subset=["feature_key", "outcome_col"]
    ).copy()

    n_exact_duplicates_removed = (
        before_exact_dedup - len(usable)
    )

    # Identify genuinely ambiguous mappings: one feature key linked to
    # multiple distinct columns in the analysis matrix.
    mapping_counts = (
        usable.groupby(
            "feature_key",
            dropna=False,
        )["outcome_col"]
        .nunique()
        .rename("n_outcome_cols")
        .reset_index()
    )

    conflicting_keys = mapping_counts.loc[
        mapping_counts["n_outcome_cols"] > 1,
        "feature_key",
    ]

    if not conflicting_keys.empty:
        conflicts = (
            usable[
                usable["feature_key"].isin(
                    conflicting_keys
                )
            ]
            .sort_values(
                ["feature_key", "outcome_col"]
            )
            .copy()
        )

        conflict_out = (
            cfg.LOG_DIR
            / f"{path.stem}_conflicting_feature_mappings.tsv"
        )

        conflicts.to_csv(
            conflict_out,
            sep="\t",
            index=False,
        )

        raise ValueError(
            f"{len(conflicting_keys):,} feature keys map to multiple "
            "outcome columns.\n"
            f"Review: {conflict_out}\n"
            "Examples:\n"
            + conflicts[
                ["feature_key", "outcome_col"]
            ]
            .head(20)
            .to_string(index=False)
        )

    # After the conflict check, every feature key has exactly one safe
    # outcome-column mapping.
    unique_map = (
        usable[
            ["feature_key", "outcome_col"]
        ]
        .drop_duplicates(
            subset=["feature_key"]
        )
        .sort_values("feature_key")
        .reset_index(drop=True)
    )

    print(f"Feature-map file: {path}")
    print(f"Available mapping rows read: {before_exact_dedup:,}")
    print(
        "Repeated identical mappings collapsed:",
        f"{n_exact_duplicates_removed:,}",
    )
    print(
        "Unique feature-to-column mappings:",
        f"{len(unique_map):,}",
    )

    return dict(
        zip(
            unique_map["feature_key"],
            unique_map["outcome_col"],
        )
    )


protein_matrix = pd.read_csv(
    protein_matrix_file,
    sep="\t",
    compression="gzip",
    low_memory=False,
)
protein_feature_map = load_feature_map(protein_map_file)

# Common disease main-effect fits for every protein target and every model-plan row.
protein_fits = aru.run_unique_replication_fits(
    matrix=protein_matrix,
    analysis_targets=targets,
    feature_column_map=protein_feature_map,
    omic_type="proteomics",
    age_col="age_at_assay",
    model_plan=model_plan,
    pc_columns=cfg.PC_COLUMNS,
    min_cases=cfg.MIN_CASES_PROTEOMICS,
    min_controls=cfg.MIN_CONTROLS_PROTEOMICS,
    age_cutoff=cfg.AGE_CUTOFF,
)
protein_fits.to_csv(
    cfg.RESULTS_DIR / "proteomics_unique_replication_fits.tsv",
    sep="\t",
    index=False,
)

# Additional discovery-concordant joint interaction for Spline protein targets.
protein_spline_fits, protein_spline_contrasts = (
    aru.run_unique_spline_interaction_fits(
        matrix=protein_matrix,
        analysis_targets=targets,
        feature_column_map=protein_feature_map,
        omic_type="proteomics",
        age_col="age_at_assay",
        primary_spec=primary_spec,
        pc_columns=cfg.PC_COLUMNS,
        min_cases=cfg.MIN_CASES_PROTEOMICS,
        min_controls=cfg.MIN_CONTROLS_PROTEOMICS,
        reference_ages=(40.0, 55.0, 70.0),
    )
)
protein_spline_fits.to_csv(
    cfg.RESULTS_DIR / "proteomics_spline_interaction_fit_links.tsv",
    sep="\t",
    index=False,
)

print("Protein common-model statuses:")
display(protein_fits["status"].value_counts(dropna=False))
print("Protein spline-interaction statuses:")
display(protein_spline_fits["status_interaction"].value_counts(dropna=False))


In [ ]:
chemistry_matrix = pd.read_csv(
    chemistry_matrix_file,
    sep="\t",
    compression="gzip",
    low_memory=False,
)
chemistry_feature_map = load_feature_map(chemistry_map_file)

# Common disease main-effect fits for every chemistry target and model-plan row.
chemistry_fits = aru.run_unique_replication_fits(
    matrix=chemistry_matrix,
    analysis_targets=targets,
    feature_column_map=chemistry_feature_map,
    omic_type="chemistries",
    age_col="age_at_measurement",
    model_plan=model_plan,
    pc_columns=cfg.PC_COLUMNS,
    min_cases=cfg.MIN_CASES_CHEMISTRY,
    min_controls=cfg.MIN_CONTROLS_CHEMISTRY,
    age_cutoff=cfg.AGE_CUTOFF,
)
chemistry_fits.to_csv(
    cfg.RESULTS_DIR / "chemistry_unique_replication_fits.tsv",
    sep="\t",
    index=False,
)

# Additional discovery-concordant joint interaction for Spline chemistry targets.
chemistry_spline_fits, chemistry_spline_contrasts = (
    aru.run_unique_spline_interaction_fits(
        matrix=chemistry_matrix,
        analysis_targets=targets,
        feature_column_map=chemistry_feature_map,
        omic_type="chemistries",
        age_col="age_at_measurement",
        primary_spec=primary_spec,
        pc_columns=cfg.PC_COLUMNS,
        min_cases=cfg.MIN_CASES_CHEMISTRY,
        min_controls=cfg.MIN_CONTROLS_CHEMISTRY,
        reference_ages=(40.0, 55.0, 70.0),
    )
)
chemistry_spline_fits.to_csv(
    cfg.RESULTS_DIR / "chemistry_spline_interaction_fit_links.tsv",
    sep="\t",
    index=False,
)

print("Chemistry common-model statuses:")
display(chemistry_fits["status"].value_counts(dropna=False))
print("Chemistry spline-interaction statuses:")
display(chemistry_spline_fits["status_interaction"].value_counts(dropna=False))


In [ ]:
# -----------------------------------------------------------------------------
# Combine common main-effect fits and attach them to discovery targets
# -----------------------------------------------------------------------------

# Fit tables contain one row per discovery feature-key link. When two feature
# keys map to the same AoU outcome, they share unique_aou_test_id; regression and
# FDR are computed only once and then copied back to both links.
fit_links = pd.concat([protein_fits, chemistry_fits], ignore_index=True, sort=False)
if fit_links["fit_id"].duplicated().any():
    raise ValueError("fit_id is not unique; inspect domain-feature-model links")

fit_links.to_csv(
    cfg.RESULTS_DIR / "combined_replication_fit_links.tsv",
    sep="\t",
    index=False,
)

unique_tests = fit_links.drop_duplicates("unique_aou_test_id").copy()
unique_tests.to_csv(
    cfg.RESULTS_DIR / "combined_unique_aou_tests.tsv",
    sep="\t",
    index=False,
)

combined = aru.attach_fits_to_targets(targets, fit_links)

# Targets without a clinical phenotype never entered the fitting set. Make their
# status explicit rather than leaving a silent blank.
no_fit = combined["status"].isna()
combined.loc[
    no_fit & ~combined["clinically_testable"].fillna(False), "status"
] = "not_clinically_testable"
combined.loc[
    no_fit & combined["clinically_testable"].fillna(False), "status"
] = "no_fit_row"

# This all-model table intentionally retains the common main-effect estimand for
# every discovery method and all sensitivity specifications.
combined.to_csv(
    cfg.RESULTS_DIR / "combined_target_replication_results_all_models.tsv",
    sep="\t",
    index=False,
)

primary_main = aru.primary_result_table(combined, cfg.PRIMARY_ANALYSIS_NAME)

# -----------------------------------------------------------------------------
# Combine discovery-concordant spline fits and fixed-age contrasts
# -----------------------------------------------------------------------------

spline_fit_links = pd.concat(
    [protein_spline_fits, chemistry_spline_fits],
    ignore_index=True,
    sort=False,
)
if (
    not spline_fit_links.empty
    and spline_fit_links["spline_fit_link_id"].duplicated().any()
):
    raise ValueError(
        "spline_fit_link_id is not unique; inspect spline domain-feature links"
    )

spline_contrasts = pd.concat(
    [protein_spline_contrasts, chemistry_spline_contrasts],
    ignore_index=True,
    sort=False,
)

spline_fit_links.to_csv(
    cfg.RESULTS_DIR / "combined_spline_interaction_fit_links.tsv",
    sep="\t",
    index=False,
)
spline_fit_links.drop_duplicates("unique_spline_test_id").to_csv(
    cfg.RESULTS_DIR / "combined_unique_spline_interaction_tests.tsv",
    sep="\t",
    index=False,
)
spline_contrasts.to_csv(
    cfg.RESULTS_DIR / "combined_spline_age_specific_contrasts.tsv",
    sep="\t",
    index=False,
)

# Enrich the primary table with the test that matches each discovery hypothesis:
# VAE/OLS -> adjusted disease main effect; Spline -> joint age interaction.
primary = aru.attach_discovery_concordant_results(
    primary_main_effect_results=primary_main,
    spline_fit_links=spline_fit_links,
)
primary.to_csv(
    cfg.RESULTS_DIR / "combined_target_replication_results_primary.tsv",
    sep="\t",
    index=False,
)

sensitivities = combined[
    ~combined["analysis_name"].eq(cfg.PRIMARY_ANALYSIS_NAME)
    & combined["analysis_name"].notna()
].copy()
sensitivities.to_csv(
    cfg.RESULTS_DIR / "combined_target_replication_results_sensitivities.tsv",
    sep="\t",
    index=False,
)

print("Common fit-link rows:", len(fit_links))
print("Unique common AoU tests across all models:", len(unique_tests))
print("All target-model rows:", len(combined))
print("Primary target rows:", len(primary))
print("Unique spline interaction tests:", spline_fit_links["unique_spline_test_id"].nunique())
print("Fixed-age spline contrast rows:", len(spline_contrasts))
print("Discovery-concordant primary statuses:")
display(primary["discovery_concordant_status"].value_counts(dropna=False))


In [ ]:
# A primary result should occur at most once per grouped discovery target.
duplicates = primary[primary["analysis_target_id"].duplicated(keep=False)]
if not duplicates.empty:
    raise ValueError(
        "Multiple primary rows were attached to the same analysis target. "
        "Inspect feature/domain keys before interpreting results."
    )

# Two summaries are retained deliberately:
# 1) common main-effect summary for direct cross-method comparability;
# 2) discovery-concordant summary, where Spline uses the joint interaction test.
main_effect_summary = aru.method_summary(primary)
main_effect_summary.to_csv(
    cfg.RESULTS_DIR / "primary_main_effect_replication_summary_by_method.tsv",
    sep="\t",
    index=False,
)

summary = aru.discovery_concordant_method_summary(primary)
summary.to_csv(
    cfg.RESULTS_DIR / "primary_replication_summary_by_method.tsv",
    sep="\t",
    index=False,
)

# Availability/status denominators are essential because VAE and univariate
# candidate sets have different AoU feature coverage and test families.
availability = (
    primary.groupby(
        [
            "discovery_framework",
            "discovery_model",
            "omic_type",
            "replication_test_family",
            "discovery_concordant_status",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(n_analysis_targets=("analysis_target_id", "nunique"))
)
availability.to_csv(
    cfg.RESULTS_DIR / "primary_target_availability_by_method.tsv",
    sep="\t",
    index=False,
)

# One denominator table spans candidate generation, clinical testability,
# feature availability, successful modeling, and concordant replication evidence.
target_denominators = (
    targets.groupby(
        ["discovery_framework", "discovery_model", "omic_type"],
        dropna=False,
        as_index=False,
    )
    .agg(
        n_total_candidate_targets=("analysis_target_id", "nunique"),
        n_clinically_testable=("clinically_testable", "sum"),
    )
)
target_denominators["n_not_clinically_testable"] = (
    target_denominators["n_total_candidate_targets"]
    - target_denominators["n_clinically_testable"]
)

status = primary["discovery_concordant_status"].astype("string")
primary_for_denominators = primary.assign(_concordant_status=status)
model_denominators = (
    primary_for_denominators.groupby(
        ["discovery_framework", "discovery_model", "omic_type"],
        dropna=False,
        as_index=False,
    )
    .agg(
        n_primary_rows=("analysis_target_id", "nunique"),
        n_feature_unavailable=(
            "_concordant_status",
            lambda values: int(values.eq("feature_not_available").sum()),
        ),
        n_modelled=(
            "_concordant_status",
            lambda values: int(values.eq("ok").sum()),
        ),
        n_low_n_age_or_df=(
            "_concordant_status",
            lambda values: int(
                values.isin(
                    [
                        "skipped_low_n",
                        "skipped_low_n_after_covariates",
                        "insufficient_residual_df",
                        "insufficient_age_variation",
                        "no_case_control_variation",
                    ]
                ).sum()
            ),
        ),
        n_model_errors=(
            "_concordant_status",
            lambda values: int(
                values.isin(["model_error", "missing_columns"]).sum()
            ),
        ),
    )
)

denominator_summary = (
    target_denominators.merge(
        model_denominators,
        on=["discovery_framework", "discovery_model", "omic_type"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        summary,
        on=["discovery_framework", "discovery_model", "omic_type"],
        how="left",
        suffixes=("", "_method_summary"),
        validate="one_to_one",
    )
)
denominator_summary.to_csv(
    cfg.RESULTS_DIR / "combined_method_denominator_summary.tsv",
    sep="\t",
    index=False,
)

print("Discovery-concordant method summary:")
display(summary)
print("Common main-effect method summary:")
display(main_effect_summary)
print("Availability by concordant test status:")
display(availability)
print("Full denominator summary:")
display(denominator_summary)


In [ ]:
# Human-readable troubleshooting view. For Spline rows, the common main-effect
# coefficient is retained, but the displayed replication evidence is the joint
# disease x spline(age) test.
view_cols = [
    "discovery_model_label",
    "omic_label",
    "analysis_domain_label",
    "feature_display",
    "prs_names",
    "discovery_direction",
    "replication_test_family",
    "beta_case",
    "p_case",
    "q_within_analysis_omic_domain",
    "interaction_statistic",
    "interaction_df",
    "p_interaction",
    "q_interaction_within_omic_domain",
    "discovery_concordant_p",
    "discovery_concordant_q",
    "discovery_concordant_replication_category",
    "discovery_concordant_n_cases",
    "discovery_concordant_n_controls",
    "discovery_concordant_status",
]
view_cols = [column for column in view_cols if column in primary.columns]
display(
    primary[view_cols].sort_values(
        ["discovery_concordant_status", "discovery_concordant_p"],
        na_position="last",
    ).head(150)
)
